# PROCESAMIENTO POBLACIÓN DANE

## 1. IMPORTACIÓN LIBRERIAS Y RUTAS

In [22]:
import pandas as pd
import numpy as np
import re
import os
import time

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

pandas  : 2.3.3
numpy   : 2.0.2


## 2. RUTAS DE ENTRADA Y SALIDA

In [23]:
# ── Rutas en Kaggle ────────────────────────────────────────────────────────────
PATH_PROYECCIONES = (
    "/kaggle/input/datasets/nicolasacostaa/poblacion-cruda/"
    "PPED-AreaSexoEdadMun-2018-2042_VP.xlsx"
)
PATH_REALES_2005_2017 = (
    "/kaggle/input/datasets/nicolasacostaa/poblacion-2005-2017/"
    "DCD-area-sexo-edad-proypoblacion-Mun-2005-2017_VP.xlsx"
)
PATH_REALES_1995_2004 = (
    "/kaggle/input/datasets/nicolasacostaa/poblacion-colombia-1995-2004/"
    "DCD-area-sexo-edad-proypoblacion-Mun-1995-2004.xlsx"
)
PATH_REALES_1985_1994 = (
    "/kaggle/input/datasets/nicolasacostaa/poblacion-colombia-1985-1994/"
    "DCD-area-sexo-edad-proypoblacion-Mun-1985-1994.xlsx"
)

OUTPUT_PATH = "/kaggle/working/df_poblacion_colombia.parquet"

# Columnas de identificación unificadas en el dataset final
ID_COLS_FINAL = [
    "DP",             # Código departamento (2 dígitos)
    "DPNOM",          # Nombre departamento
    "COD_MPIO",       # Código municipio (5 dígitos)
    "NOM_MPIO",       # Nombre municipio
    "AÑO",
    "ÁREA GEOGRÁFICA",
]

## 3. FUNCIONES AUXILIARES

### 3.1. EXTRAER EDAD

In [24]:
def extraer_edad(nombre_col: str) -> int:
    """
    Extrae el primer número entero encontrado en el nombre de una columna de edad.

    Ejemplos:
        'Hombres 0 años'         -> 0
        'Hombres 100 años y más' -> 100
        'Hombres_85 y más'       -> 85
        'Mujeres_3'              -> 3
    """
    numeros = re.findall(r"\d+", nombre_col)
    return int(numeros[0]) if numeros else np.nan

### 3.2. DETECTAR COLUMNAS EDAD

In [25]:
def detectar_columnas_edad(columnas: list, prefijo: str) -> list:
    """
    Devuelve las columnas que corresponden a edades simples para un género dado.
    Excluye los subtotales como 'Hombres', 'Total Hombres', etc.

    Args:
        columnas: lista de nombres de columnas del DataFrame.
        prefijo  : 'Hombres' o 'Mujeres' (insensible a mayúsculas).
    """
    patron = re.compile(
        rf"^{prefijo}[_ ]\d",   # Hombres_0 / Hombres 0 años
        re.IGNORECASE
    )
    return [c for c in columnas if patron.match(c)]

### 3.3. PIVOTAR GÉNERO

In [26]:
def pivotar_genero(df: pd.DataFrame, id_cols: list, cols_genero: list,
                   etiqueta_genero: str) -> pd.DataFrame:
    """
    Convierte las columnas de un género (wide) a formato largo.

    Retorna un DataFrame con columnas:
        [id_cols] + GENERO + EDAD EN ANIOS + POBLACION
    """
    df_largo = (
        df[id_cols + cols_genero]
        .melt(
            id_vars=id_cols,
            value_vars=cols_genero,
            var_name="_col_edad",
            value_name="POBLACION",
        )
    )
    df_largo["GENERO"] = etiqueta_genero
    df_largo["EDAD EN ANIOS"] = df_largo["_col_edad"].apply(extraer_edad)
    df_largo = df_largo.drop(columns="_col_edad")
    return df_largo

### 3.4. TRANSFORMAR A LARGO

In [27]:
def transformar_a_largo(df: pd.DataFrame, id_cols: list) -> pd.DataFrame:
    """
    Aplica la transformación completa wide → long a un DataFrame ya cargado:
      - Elimina filas de ÁREA GEOGRÁFICA == 'Total' (son sumas calculadas).
      - Extrae columnas de Hombres y Mujeres por edad individual.
      - Genera los campos GENERO, EDAD EN ANIOS y POBLACION.
      - Elimina columnas de totales (Total_X, Total Hombres, etc.).

    Args:
        df      : DataFrame en formato wide con columnas de edades.
        id_cols : Lista de columnas de identificación a conservar.

    Returns:
        DataFrame largo normalizado.
    """
    # Filtrar totales territoriales (filas donde área es 'Total')
    df = df[df["ÁREA GEOGRÁFICA"].str.strip() != "Total"].copy()

    cols_hombres = detectar_columnas_edad(df.columns.tolist(), "Hombres")
    cols_mujeres = detectar_columnas_edad(df.columns.tolist(), "Mujeres")

    df_h = pivotar_genero(df, id_cols, cols_hombres, "Hombre")
    df_m = pivotar_genero(df, id_cols, cols_mujeres, "Mujer")

    df_largo = pd.concat([df_h, df_m], ignore_index=True)

    # Eliminar nulos en POBLACION o edad (pueden venir de celdas vacías)
    df_largo = df_largo.dropna(subset=["POBLACION", "EDAD EN ANIOS"])

    # Tipos finales
    df_largo["EDAD EN ANIOS"] = df_largo["EDAD EN ANIOS"].astype(int)
    df_largo["POBLACION"] = df_largo["POBLACION"].astype(int)

    return df_largo

## 4. CARGA ARCHIVO PROYECCIONES

In [28]:
def cargar_proyecciones(path: str) -> pd.DataFrame:
    """
    Carga y transforma el archivo de proyecciones DANE (2018-2042).

    Estructura del archivo:
        - Hoja : 'PobMunicipalxÁreaSexoEdad'
        - Fila 8 (idx 7) : grupo de encabezados (DP, DPNOM, MPIO, DPMP, AÑO, ÁREA, …)
        - Fila 9 (idx 8) : nombres de columna detallados (None para las 6 primeras)
        - Datos a partir de la fila 10 (idx 9)
        - Edades: 'Hombres 0 años' … 'Hombres 100 años y más'
        - MPIO contiene el código de 5 dígitos, DPMP contiene el nombre

    Returns:
        DataFrame largo con columnas: ID_COLS_FINAL + GENERO + EDAD EN ANIOS + POBLACION.
    """
    print("[Proyecciones] Leyendo encabezados...")
    HOJA = "PobMunicipalxÁreaSexoEdad"

    # Leer las dos filas de encabezado para reconstruir nombres de columna
    hdrs = pd.read_excel(
        path, sheet_name=HOJA, skiprows=7, nrows=2, header=None
    )

    # Combinar filas de encabezado: prioridad a la fila 2 (índice 1)
    nombres_col = []
    for i in range(hdrs.shape[1]):
        val_fila2 = hdrs.iloc[1, i]
        val_fila1 = hdrs.iloc[0, i]
        if pd.notna(val_fila2):
            nombres_col.append(str(val_fila2).strip())
        elif pd.notna(val_fila1):
            nombres_col.append(str(val_fila1).strip())
        else:
            nombres_col.append(f"_col_{i}")

    print(f"[Proyecciones] Total columnas detectadas: {len(nombres_col)}")

    # Leer datos (a partir de la fila 10, índice 9)
    print("[Proyecciones] Cargando datos (puede tardar unos minutos)...")
    t0 = time.time()
    df_raw = pd.read_excel(
        path,
        sheet_name=HOJA,
        skiprows=9,
        header=None,
        names=nombres_col,
        dtype={"MPIO": str, "DP": str},  # preservar ceros a la izquierda
    )
    print(f"[Proyecciones] Leídas {len(df_raw):,} filas en {time.time()-t0:.1f}s")

    # ── Renombrar para homologar con archivos reales ──────────────────────────
    # En este archivo: col 3='MPIO' tiene el código; col 4='DPMP' tiene el nombre
    df_raw = df_raw.rename(columns={"MPIO": "COD_MPIO", "DPMP": "NOM_MPIO"})

    # Eliminar filas completamente vacías (posibles footers del Excel)
    df_raw = df_raw.dropna(subset=["COD_MPIO", "AÑO"])

    id_cols_origen = ["DP", "DPNOM", "COD_MPIO", "NOM_MPIO", "AÑO", "ÁREA GEOGRÁFICA"]

    # Transformar a formato largo
    print("[Proyecciones] Transformando a formato largo...")
    df_largo = transformar_a_largo(df_raw, id_cols_origen)
    df_largo["TIPO"] = 1  # Proyección

    print(f"[Proyecciones] Filas resultantes: {len(df_largo):,}")
    print(f"[Proyecciones] Años: {sorted(df_largo['AÑO'].unique())}\n")
    return df_largo


df_proyecciones = cargar_proyecciones(PATH_PROYECCIONES)
df_proyecciones.head()

[Proyecciones] Leyendo encabezados...
[Proyecciones] Total columnas detectadas: 312
[Proyecciones] Cargando datos (puede tardar unos minutos)...
[Proyecciones] Leídas 84,233 filas en 333.9s
[Proyecciones] Transformando a formato largo...
[Proyecciones] Filas resultantes: 11,342,300
[Proyecciones] Años: [np.float64(2018.0), np.float64(2019.0), np.float64(2020.0), np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0), np.float64(2025.0), np.float64(2026.0), np.float64(2027.0), np.float64(2028.0), np.float64(2029.0), np.float64(2030.0), np.float64(2031.0), np.float64(2032.0), np.float64(2033.0), np.float64(2034.0), np.float64(2035.0), np.float64(2036.0), np.float64(2037.0), np.float64(2038.0), np.float64(2039.0), np.float64(2040.0), np.float64(2041.0), np.float64(2042.0)]



,DP,DPNOM,COD_MPIO,NOM_MPIO,AÑO,ÁREA GEOGRÁFICA,POBLACION,GENERO,EDAD EN ANIOS,TIPO
0,05,Antioquia,05001,Medellín,2018.0,Cabecera Municipal,13512,Hombre,0,1
1,05,Antioquia,05001,Medellín,2018.0,Centros Poblados y Rural Disperso,304,Hombre,0,1
2,05,Antioquia,05002,Abejorral,2018.0,Cabecera Municipal,38,Hombre,0,1
3,05,Antioquia,05002,Abejorral,2018.0,Centros Poblados y Rural Disperso,96,Hombre,0,1
4,05,Antioquia,05004,Abriaquí,2018.0,Cabecera Municipal,8,Hombre,0,1


## 5. CARGA ARCHIVOS DE DATOS REALES (1985-2017)

In [29]:
def cargar_reales(path: str, nombre_hoja: str) -> pd.DataFrame:
    """
    Carga y transforma un archivo de datos reales DANE (1985-2017).

    Estructura del archivo:
        - Encabezado en fila 12 (idx 11), datos desde fila 13.
        - Edades: 'Hombres_0' … 'Hombres_85 y más'.

    IMPORTANTE — inconsistencia entre archivos fuente:
        · 2005-2017 (NuevaMpal) : DPMP = nombre municipio, MPIO = código 5 dígitos.
        · 1985-1994 y 1995-2004 (Municipal): DPMP = código 5 dígitos, MPIO = nombre.
    La función detecta automáticamente qué columna contiene el código numérico
    y renombra en consecuencia, evitando valores corruptos (ej. '0Anzá').

    Args:
        path        : Ruta al archivo Excel.
        nombre_hoja : Nombre de la hoja a leer.

    Returns:
        DataFrame largo con columnas: ID_COLS_FINAL + GENERO + EDAD EN ANIOS + POBLACION.
    """
    print(f"[Reales] Cargando: {os.path.basename(path)} / hoja='{nombre_hoja}'")
    t0 = time.time()

    df_raw = pd.read_excel(
        path,
        sheet_name=nombre_hoja,
        skiprows=11,      # omitir filas 1–11; fila 12 es el encabezado
        header=0,
        dtype={"MPIO": str, "DPMP": str, "DP": str},
    )
    print(f"  Leídas {len(df_raw):,} filas en {time.time()-t0:.1f}s")

    # ── Auto-detectar orientación de columnas DPMP / MPIO ────────────────────
    # Si DPMP contiene solo dígitos en su primera celda válida → es el código.
    # De lo contrario (contiene texto) → es el nombre y MPIO es el código.
    primer_dpmp = str(df_raw["DPMP"].dropna().iloc[0]).strip()
    if primer_dpmp.isdigit():
        # Archivos 1985-1994 y 1995-2004: DPMP=código, MPIO=nombre
        df_raw = df_raw.rename(columns={"DPMP": "COD_MPIO", "MPIO": "NOM_MPIO"})
        print("  Orientación detectada: DPMP=código / MPIO=nombre")
    else:
        # Archivo 2005-2017: DPMP=nombre, MPIO=código
        df_raw = df_raw.rename(columns={"MPIO": "COD_MPIO", "DPMP": "NOM_MPIO"})
        print("  Orientación detectada: MPIO=código / DPMP=nombre")

    # Eliminar filas vacías
    df_raw = df_raw.dropna(subset=["COD_MPIO", "AÑO"])

    id_cols_origen = ["DP", "DPNOM", "COD_MPIO", "NOM_MPIO", "AÑO", "ÁREA GEOGRÁFICA"]

    # Transformar a formato largo
    print("  Transformando a formato largo...")
    df_largo = transformar_a_largo(df_raw, id_cols_origen)
    df_largo["TIPO"] = 0  # Real / actual

    print(f"  Filas resultantes: {len(df_largo):,}")
    print(f"  Años: {sorted(df_largo['AÑO'].unique())}\n")
    return df_largo

In [30]:
# Cargar los tres períodos de datos reales
df_r_2005_2017 = cargar_reales(PATH_REALES_2005_2017, nombre_hoja="NuevaMpal")
df_r_1995_2004 = cargar_reales(PATH_REALES_1995_2004, nombre_hoja="Municipal")
df_r_1985_1994 = cargar_reales(PATH_REALES_1985_1994, nombre_hoja="Municipal")

df_r_2005_2017.head()

[Reales] Cargando: DCD-area-sexo-edad-proypoblacion-Mun-2005-2017_VP.xlsx / hoja='NuevaMpal'
  Leídas 43,758 filas en 137.4s
  Orientación detectada: MPIO=código / DPMP=nombre
  Transformando a formato largo...
  Filas resultantes: 5,017,584
  Años: [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]

[Reales] Cargando: DCD-area-sexo-edad-proypoblacion-Mun-1995-2004.xlsx / hoja='Municipal'
  Leídas 33,660 filas en 105.1s
  Orientación detectada: DPMP=código / MPIO=nombre
  Transformando a formato largo...
  Filas resultantes: 3,859,680
  Años: [np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004)]

[Reales] Cargando: DCD-area-sexo-edad-proypoblacion-Mun-1985-1994.xlsx / hoja='Municipal'
  Leídas 33,660 filas en 101.1s
  Orienta

,DP,DPNOM,COD_MPIO,NOM_MPIO,AÑO,ÁREA GEOGRÁFICA,POBLACION,GENERO,EDAD EN ANIOS,TIPO
0,05,Antioquia,05001,Medellín,2005,Cabecera Municipal,13812,Hombre,0,0
1,05,Antioquia,05001,Medellín,2005,Centros Poblados y Rural Disperso,489,Hombre,0,0
2,05,Antioquia,05001,Medellín,2006,Cabecera Municipal,13674,Hombre,0,0
3,05,Antioquia,05001,Medellín,2006,Centros Poblados y Rural Disperso,475,Hombre,0,0
4,05,Antioquia,05001,Medellín,2007,Cabecera Municipal,13469,Hombre,0,0


## 6. COMBINACIÓN DE DATOS REALES+PROYECCIONES

In [31]:
def combinar_datasets(*dfs: pd.DataFrame) -> pd.DataFrame:
    """
    Une múltiples DataFrames normalizados en un único dataset.
    Garantiza homologación de tipos y orden de columnas.

    Columnas finales:
        DP · DPNOM · COD_MPIO · NOM_MPIO · AÑO · ÁREA GEOGRÁFICA
        · GENERO · EDAD EN ANIOS · POBLACION · TIPO
    """
    COLUMNAS_ORDEN = [
        "DP", "DPNOM", "COD_MPIO", "NOM_MPIO",
        "AÑO", "ÁREA GEOGRÁFICA",
        "GENERO", "EDAD EN ANIOS", "POBLACION", "TIPO",
    ]

    df_total = pd.concat(list(dfs), ignore_index=True)

    # ── Homologar tipos ───────────────────────────────────────────────────────
    df_total["DP"]            = df_total["DP"].astype(str).str.zfill(2)
    df_total["COD_MPIO"]      = df_total["COD_MPIO"].astype(str).str.zfill(5)
    df_total["AÑO"]           = df_total["AÑO"].astype(int)
    df_total["EDAD EN ANIOS"] = df_total["EDAD EN ANIOS"].astype(int)
    df_total["POBLACION"]     = df_total["POBLACION"].astype(int)
    df_total["TIPO"]          = df_total["TIPO"].astype(int)

    # ── Normalizar texto ──────────────────────────────────────────────────────
    df_total["DPNOM"]           = df_total["DPNOM"].str.strip()
    df_total["NOM_MPIO"]        = df_total["NOM_MPIO"].str.strip()
    df_total["ÁREA GEOGRÁFICA"] = df_total["ÁREA GEOGRÁFICA"].str.strip()
    df_total["GENERO"]          = df_total["GENERO"].str.strip()

    return df_total[COLUMNAS_ORDEN].sort_values(
        ["DP", "COD_MPIO", "AÑO", "ÁREA GEOGRÁFICA", "GENERO", "EDAD EN ANIOS"]
    ).reset_index(drop=True)


df_poblacion = combinar_datasets(
    df_r_1985_1994,
    df_r_1995_2004,
    df_r_2005_2017,
    df_proyecciones,
)

print("\n=== Resumen del dataset combinado ===")
print(f"  Filas totales  : {len(df_poblacion):,}")
print(f"  Columnas       : {df_poblacion.columns.tolist()}")
print(f"  Años cubiertos : {df_poblacion['AÑO'].min()} – {df_poblacion['AÑO'].max()}")
print(f"  Municipios     : {df_poblacion['COD_MPIO'].nunique():,}")
print(f"  Distribución TIPO:")
print(df_poblacion['TIPO'].value_counts().rename({0: 'Real (0)', 1: 'Proyección (1)'}))
df_poblacion.head(10)


=== Resumen del dataset combinado ===
  Filas totales  : 24,079,244
  Columnas       : ['DP', 'DPNOM', 'COD_MPIO', 'NOM_MPIO', 'AÑO', 'ÁREA GEOGRÁFICA', 'GENERO', 'EDAD EN ANIOS', 'POBLACION', 'TIPO']
  Años cubiertos : 1985 – 2042
  Municipios     : 1,123
  Distribución TIPO:
TIPO
Real (0)          12736944
Proyección (1)    11342300
Name: count, dtype: int64


,DP,DPNOM,COD_MPIO,NOM_MPIO,AÑO,ÁREA GEOGRÁFICA,GENERO,EDAD EN ANIOS,POBLACION,TIPO
0,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,0,13945,0
1,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,1,13868,0
2,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,2,13854,0
3,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,3,13822,0
4,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,4,13702,0
5,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,5,13430,0
6,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,6,13048,0
7,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,7,12653,0
8,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,8,12251,0
9,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,9,11992,0


## 7. VALIDACIÓN DE INTEGRIDAD

In [32]:
def validar_dataset(df: pd.DataFrame) -> None:
    """
    Ejecuta controles básicos de calidad sobre el dataset final.
    Imprime alertas si se detectan anomalías.
    """
    print("=== Validación de calidad ===")

    # 1. Nulos
    nulos = df.isnull().sum()
    nulos_reales = nulos[nulos > 0]
    if nulos_reales.empty:
        print("  ✓ Sin valores nulos")
    else:
        print(f"  ⚠ Nulos detectados:\n{nulos_reales}")

    # 2. TIPO solo contiene 0 y 1
    tipos_invalidos = df[~df["TIPO"].isin([0, 1])]
    if tipos_invalidos.empty:
        print("  ✓ TIPO solo contiene valores 0 y 1")
    else:
        print(f"  ⚠ TIPO con valores inesperados: {tipos_invalidos['TIPO'].unique()}")

    # 3. GENERO solo Hombre / Mujer
    generos = df["GENERO"].unique()
    if set(generos) == {"Hombre", "Mujer"}:
        print("  ✓ GENERO contiene solo 'Hombre' y 'Mujer'")
    else:
        print(f"  ⚠ Valores inesperados en GENERO: {generos}")

    # 4. Edades no negativas
    edades_neg = df[df["EDAD EN ANIOS"] < 0]
    if edades_neg.empty:
        print(f"  ✓ Edades en rango válido: [{df['EDAD EN ANIOS'].min()} – {df['EDAD EN ANIOS'].max()}]")
    else:
        print(f"  ⚠ Edades negativas: {len(edades_neg)} filas")

    # 5. POBLACION no negativa
    pob_neg = df[df["POBLACION"] < 0]
    if pob_neg.empty:
        print("  ✓ Sin valores de POBLACION negativos")
    else:
        print(f"  ⚠ POBLACION negativa: {len(pob_neg)} filas")

    # 6. Solapamiento de años entre TIPO=0 y TIPO=1
    anios_real = set(df[df["TIPO"] == 0]["AÑO"].unique())
    anios_proy = set(df[df["TIPO"] == 1]["AÑO"].unique())
    solapamiento = anios_real & anios_proy
    if not solapamiento:
        print("  ✓ Sin solapamiento de años entre datos reales y proyecciones")
    else:
        print(f"  ⚠ Años solapados: {sorted(solapamiento)}")

    # 7. Suma de Hombre + Mujer por municipio/año/área/edad (consistencia interna)
    print("\n  Muestra de suma H+M para Medellín 2020 (primer año proyección):")
    muestra = (
        df[
            (df["COD_MPIO"] == "05001")
            & (df["AÑO"] == 2020)
            & (df["ÁREA GEOGRÁFICA"] == "Cabecera Municipal")
            & (df["EDAD EN ANIOS"] <= 5)
        ]
        .pivot_table(
            index="EDAD EN ANIOS",
            columns="GENERO",
            values="POBLACION",
            aggfunc="sum",
        )
    )
    if not muestra.empty:
        muestra["TOTAL"] = muestra.sum(axis=1)
        print(muestra)
    else:
        print("  (Sin datos para esa combinación — verifique el código de municipio)")


validar_dataset(df_poblacion)

=== Validación de calidad ===
  ✓ Sin valores nulos
  ✓ TIPO solo contiene valores 0 y 1
  ✓ GENERO contiene solo 'Hombre' y 'Mujer'
  ✓ Edades en rango válido: [0 – 100]
  ✓ Sin valores de POBLACION negativos
  ✓ Sin solapamiento de años entre datos reales y proyecciones

  Muestra de suma H+M para Medellín 2020 (primer año proyección):
GENERO         Hombre  Mujer  TOTAL
EDAD EN ANIOS                      
0               13432  12789  26221
1               13700  13036  26736
2               14148  13478  27626
3               14490  13807  28297
4               14767  14157  28924
5               15058  14450  29508


## 8. EXPORTACIÓN A PARQUET

In [33]:
def exportar_parquet(df: pd.DataFrame, output_path: str) -> None:
    """
    Exporta el DataFrame final al formato Parquet.

    Usa compresión 'snappy' (predeterminada en pyarrow) para equilibrar
    tamaño en disco y velocidad de lectura.

    Args:
        df          : DataFrame a exportar.
        output_path : Ruta completa del archivo .parquet resultante.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    print(f"Exportando a: {output_path}")
    t0 = time.time()
    df.to_parquet(output_path, index=False, engine="pyarrow", compression="snappy")
    tam_mb = os.path.getsize(output_path) / 1_048_576
    print(f"  ✓ Archivo guardado: {tam_mb:.1f} MB en {time.time()-t0:.1f}s")


exportar_parquet(df_poblacion, OUTPUT_PATH)

Exportando a: /kaggle/working/df_poblacion_colombia.parquet
  ✓ Archivo guardado: 26.3 MB en 22.5s


## 9. LECTURA DE VERIFICACIÓN DEL PARQUET

In [34]:
def verificar_parquet(output_path: str) -> None:
    """
    Lee el archivo Parquet generado y muestra un resumen para confirmar
    que fue escrito correctamente.
    """
    df_check = pd.read_parquet(output_path)
    print("=== Verificación del archivo Parquet ===")
    print(f"  Filas    : {len(df_check):,}")
    print(f"  Columnas : {df_check.columns.tolist()}")
    print()
    print(df_check.dtypes)
    print()
    print("Primeras 5 filas:")
    display(df_check.head())
    print("\nÚltimas 5 filas:")
    display(df_check.tail())


verificar_parquet(OUTPUT_PATH)

=== Verificación del archivo Parquet ===
  Filas    : 24,079,244
  Columnas : ['DP', 'DPNOM', 'COD_MPIO', 'NOM_MPIO', 'AÑO', 'ÁREA GEOGRÁFICA', 'GENERO', 'EDAD EN ANIOS', 'POBLACION', 'TIPO']

DP                 object
DPNOM              object
COD_MPIO           object
NOM_MPIO           object
AÑO                 int64
ÁREA GEOGRÁFICA    object
GENERO             object
EDAD EN ANIOS       int64
POBLACION           int64
TIPO                int64
dtype: object

Primeras 5 filas:


,DP,DPNOM,COD_MPIO,NOM_MPIO,AÑO,ÁREA GEOGRÁFICA,GENERO,EDAD EN ANIOS,POBLACION,TIPO
0,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,0,13945,0
1,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,1,13868,0
2,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,2,13854,0
3,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,3,13822,0
4,05,Antioquia,05001,Medellín,1985,Cabecera Municipal,Hombre,4,13702,0



Últimas 5 filas:


,DP,DPNOM,COD_MPIO,NOM_MPIO,AÑO,ÁREA GEOGRÁFICA,GENERO,EDAD EN ANIOS,POBLACION,TIPO
24079239,99,Vichada,99773,Cumaribo,2042,Centros Poblados y Rural Disperso,Mujer,96,2,1
24079240,99,Vichada,99773,Cumaribo,2042,Centros Poblados y Rural Disperso,Mujer,97,1,1
24079241,99,Vichada,99773,Cumaribo,2042,Centros Poblados y Rural Disperso,Mujer,98,1,1
24079242,99,Vichada,99773,Cumaribo,2042,Centros Poblados y Rural Disperso,Mujer,99,0,1
24079243,99,Vichada,99773,Cumaribo,2042,Centros Poblados y Rural Disperso,Mujer,100,7,1
